In [1]:
import pandas as pd
import numpy as np
import google.generativeai as genai

ModuleNotFoundError: No module named 'pandas'

In [ ]:
jobs = pd.read_csv("Jobs_Dataset.csv")

In [ ]:
jobs.head()

,job_id,job_title,job_description,skills_required,certifications,industry,experience_level,education_required
0,J001,Software Engineer,Designs develops and maintains software applic...,"Python, Java, Data Structures, Algorithms, Git...","AWS Certified Developer, Oracle Java Certifica...",IT & Technology,Mid,Bachelor's in Computer Science
1,J002,Frontend Developer,Builds user-facing web interfaces using modern...,"HTML, CSS, JavaScript, React, TypeScript, Resp...","Meta Front-End Developer Certificate, freeCode...",IT & Technology,Mid,Bachelor's in CS or related
2,J003,Backend Developer,Develops server-side logic databases and APIs.,"Node.js, Python, Java, SQL, NoSQL, REST, Micro...","AWS Certified Developer, MongoDB Associate",IT & Technology,Mid,Bachelor's in CS
3,J004,Full Stack Developer,Handles both frontend and backend development ...,"React, Node.js, MongoDB, SQL, REST APIs, Git, ...","AWS Developer, Full Stack Web Dev Certificate",IT & Technology,Mid,Bachelor's in CS or related
4,J005,Data Scientist,Analyzes complex data to extract insights and ...,"Python, R, Machine Learning, Statistics, Panda...","Google Data Analytics Certificate, IBM Data Sc...",IT & Technology,Mid,Master's in Data Science or Statistics


In [ ]:
jobs.isna().sum()

job_id                0
job_title             0
job_description       0
skills_required       0
certifications        0
industry              0
experience_level      0
education_required    0
dtype: int64

In [ ]:
jobs.duplicated().sum()

np.int64(0)

In [ ]:
jobs["job_profile"] = (
    jobs["job_title"].astype(str) + " " +
    jobs["job_description"].astype(str) + " " +
    jobs["skills_required"].astype(str) + " " +
    jobs["industry"].astype(str) + " " +
    jobs["education_required"].astype(str)
)

In [ ]:
jobs['job_profile']

0       Software Engineer Designs develops and maintai...
1       Frontend Developer Builds user-facing web inte...
2       Backend Developer Develops server-side logic d...
3       Full Stack Developer Handles both frontend and...
4       Data Scientist Analyzes complex data to extrac...
                              ...                        
1015    Clinical Trial Auditor Audits clinical researc...
1016    Scientific Affairs Manager Manages scientific ...
1017    Plant Biotechnologist Applies genetic and mole...
1018    Cosmetic Surgeon Performs elective surgical pr...
1019    Digital Twin Consultant Advises organizations ...
Name: job_profile, Length: 1020, dtype: object

In [ ]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
job_embeddings = model.encode(jobs["job_profile"].tolist(), convert_to_tensor=True)

In [ ]:
from sentence_transformers import util
for i, row in jobs.iterrows():
    course_embedding = model.encode(row["skills_required"], convert_to_tensor=True)
    similarities = util.cos_sim(course_embedding, job_embeddings)[0]

    top_k = 5
    top_results = similarities.topk(k=top_k)

    print(f"Skills: {row['skills_required']}")
    print(f"Expected: {row['job_title']}")
    for score, idx in zip(top_results.values, top_results.indices):
        print(f"  {score:.4f} → {jobs.iloc[int(idx)]['job_title']}")
    print("-" * 50)

Skills: Python, Java, Data Structures, Algorithms, Git, REST APIs, SQL, Problem Solving
Expected: Software Engineer
  0.5043 → Augmented Analytics Developer
  0.4951 → Streaming Data Engineer
  0.4756 → API Developer
  0.4539 → Software Engineer
  0.4436 → IoT Solutions Architect
--------------------------------------------------
Skills: HTML, CSS, JavaScript, React, TypeScript, Responsive Design, Git, UX Basics
Expected: Frontend Developer
  0.5523 → Frontend Developer
  0.4684 → Full Stack Developer
  0.4277 → Decentralized Application Developer
  0.4163 → Web Designer
  0.4067 → E-Learning Developer
--------------------------------------------------
Skills: Node.js, Python, Java, SQL, NoSQL, REST, Microservices, Docker
Expected: Backend Developer
  0.5597 → Backend Developer
  0.4758 → API Developer
  0.4755 → Streaming Data Engineer
  0.4702 → Decentralized Application Developer
  0.4458 → IoT Solutions Architect
--------------------------------------------------
Skills: React, Nod

In [ ]:
def predict_jobs_from_skills(skill_list, top_n=10):
    """
    Given a list of skills, predict the top matching jobs.
    
    Parameters:
        skill_list (list): A list of skills, e.g. ["Python", "SQL", "Machine Learning"]
        top_n (int): Number of top job matches to return (default: 10)
    
    Returns:
        DataFrame with top matching jobs, their scores, and details.
    """
    query_text = "Skills: " + ", ".join(skill_list)
    query_embedding = model.encode([query_text], convert_to_tensor=True)
    
    scores = util.cos_sim(query_embedding, job_embeddings)[0].cpu().numpy()
    jobs_copy = jobs.copy()
    jobs_copy["match_score"] = scores
    
    top_matches = jobs_copy.sort_values("match_score", ascending=False).head(top_n)
    result = top_matches[["job_title", "match_score", "skills_required", "industry", "experience_level", "education_required"]].reset_index(drop=True)
    
    print(f"\n🔍 Your Skills: {', '.join(skill_list)}")
    print("=" * 60)
    for i, row in result.iterrows():
        print(f"  {i+1}. {row['job_title']} (Score: {row['match_score']:.4f})")
        print(f"     Skills Required: {row['skills_required']}")
        print(f"     Industry: {row['industry']} | Level: {row['experience_level']} | Education: {row['education_required']}")
        print()
    
    return result


# ====== ENTER YOUR SKILLS HERE ======
# my_skills = ["Digital Workforce Foundation: RPA bots bridging legacy systems as a non-invasive integration layer", "Data Democratization & Quality: RPA bots for extracting, standardizing, and consolidating data from disparate sources", "Accelerated Implementation & ROI for RPA: Faster implementations, less coding, quicker and measurable return on investment", "Enhanced Customer & Employee Experience through RPA: Automating back-office, speeding up customer processes, reducing employee burnout"]

my_skills = ["Economics", "Finance", "Economic honours", "Visualization"] 
results = predict_jobs_from_skills(my_skills, top_n=5)



🔍 Your Skills: Economics, Finance, Economic honours, Visualization
  1. Economist (Score: 0.6326)
     Skills Required: Econometrics, Statistical Modeling, Policy Analysis, R, STATA
     Industry: Business & Management | Level: Senior | Education: Master's or PhD in Economics

  2. Foreign Exchange Trader (Score: 0.6214)
     Skills Required: Forex Markets, Technical Analysis, Risk Management, Bloomberg, Trading Platforms
     Industry: Finance & Banking | Level: Mid | Education: Bachelor's in Finance or Economics

  3. Quantitative Analyst (Score: 0.5668)
     Skills Required: Python, R, MATLAB, Stochastic Calculus, Statistics, Machine Learning
     Industry: Finance & Banking | Level: Senior | Education: Master's or PhD in Math Physics or Finance

  4. Derivatives Trader (Score: 0.5655)
     Skills Required: Options Pricing, Futures, Risk Management, Bloomberg, Quantitative Analysis
     Industry: Finance & Banking | Level: Senior | Education: Bachelor's or Master's in Finance or Ma